<a href="https://colab.research.google.com/github/nishshanka20/NLP_Test/blob/main/Real_fake_classification_using_Spacy_wordEmbedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import spacy

In [1]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 2.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
nlp=spacy.load("en_core_web_lg")

In [4]:
doc=nlp("dog cat banana king")

for token in doc:
  print(token.text,"Vecor",token.has_vector,"OOV",token.is_oov)

dog Vecor True OOV False
cat Vecor True OOV False
banana Vecor True OOV False
king Vecor True OOV False


In [5]:
doc[0].vector.shape

(300,)

In [6]:
base_token=nlp("bus")
base_token.vector.shape

(300,)

In [7]:
doc=nlp("car bus van bicycle house superman dog tesla")
for token in doc:
  print(f"{token.text}-->{base_token.text}",base_token.similarity(token))

car-->bus 0.5259357329196761
bus-->bus 1.0
van-->bus 0.2551551056402354
bicycle-->bus 0.5122272511915734
house-->bus 0.16266236625233302
superman-->bus 0.05693966150794956
dog-->bus 0.10744631775852782
tesla-->bus 0.1189610538196125


In [8]:
def print_similarity(base_word,words_to_compare):
  base_token=nlp(base_word)
  doc=nlp(words_to_compare)
  for token in doc:
    print(f"{token.text}-->{base_token.text}",base_token.similarity(token))

In [9]:
print_similarity("iphone","apple samsung iphone dog kitten")

apple-->iphone 0.4387907401919904
samsung-->iphone 0.670859081425417
iphone-->iphone 1.0
dog-->iphone 0.08211864228011527
kitten-->iphone 0.10222317834969896


In [11]:
import pandas as pd
fake_data=pd.read_csv("Fake.csv")
true_data=pd.read_csv("True.csv")

In [12]:
fake_data['label']="Fake"
true_data['label']="True"
fake_data['label_num']=0
true_data['label_num']=1

In [13]:
# concatinate datasets
df=pd.concat([fake_data,true_data],axis=0)
df=df.sample(frac=1).reset_index(drop=True)
df.head()

,title,text,subject,date,label,label_num
0,MEDIA SPIN ALERT! FBI Does a “Predawn Raid” on...,Why is the news just breaking that former Trum...,politics,"Aug 9, 2017",Fake,0
1,"Trump, Sanders fans share hunger for campaign ...",BOSTON/NEW YORK (Reuters) - Supporters of U.S....,politicsNews,"May 11, 2016",True,1
2,BREAKING NEWS: Facebook Killer Dead…Here Are T...,"MI, PA, OH and NY residents were all warned he...",politics,"Apr 18, 2017",Fake,0
3,House committee postpones hearing on Puerto Rico,NEW YORK (Reuters) - The U.S. House of Represe...,politicsNews,"October 19, 2017",True,1
4,German Social Democrats under pressure to form...,BERLIN (Reuters) - The leader of Germany s Soc...,worldnews,"November 23, 2017",True,1


In [14]:
df=df.drop(["subject","date"],axis=1)

In [15]:
df.head()

,title,text,label,label_num
0,MEDIA SPIN ALERT! FBI Does a “Predawn Raid” on...,Why is the news just breaking that former Trum...,Fake,0
1,"Trump, Sanders fans share hunger for campaign ...",BOSTON/NEW YORK (Reuters) - Supporters of U.S....,True,1
2,BREAKING NEWS: Facebook Killer Dead…Here Are T...,"MI, PA, OH and NY residents were all warned he...",Fake,0
3,House committee postpones hearing on Puerto Rico,NEW YORK (Reuters) - The U.S. House of Represe...,True,1
4,German Social Democrats under pressure to form...,BERLIN (Reuters) - The leader of Germany s Soc...,True,1


In [16]:
df.shape

(44898, 4)

In [17]:
df.label.value_counts()

,count
label,
Fake,23481
True,21417


In [20]:
doc=nlp("MEDIA SPIN ALERT! FBI Does a “Predawn Raid”")
doc.vector.shape

(300,)

In [21]:
df['vector']=df['title'].apply(lambda x:nlp(x).vector)

In [22]:
df.head()

,title,text,label,label_num,vector
0,MEDIA SPIN ALERT! FBI Does a “Predawn Raid” on...,Why is the news just breaking that former Trum...,Fake,0,"[-1.532833, 1.0841752, -2.6733406, 0.27630457,..."
1,"Trump, Sanders fans share hunger for campaign ...",BOSTON/NEW YORK (Reuters) - Supporters of U.S....,True,1,"[-2.3847966, 0.50449944, -2.5623088, 2.7246003..."
2,BREAKING NEWS: Facebook Killer Dead…Here Are T...,"MI, PA, OH and NY residents were all warned he...",Fake,0,"[2.0549786, -2.6252618, 3.0021858, -1.5666522,..."
3,House committee postpones hearing on Puerto Rico,NEW YORK (Reuters) - The U.S. House of Represe...,True,1,"[-2.1834514, 0.6499029, -3.4027884, 0.12262814..."
4,German Social Democrats under pressure to form...,BERLIN (Reuters) - The leader of Germany s Soc...,True,1,"[-3.1101947, 1.1004223, -1.988578, 1.8080444, ..."


In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    df.vector.values,
    df.label_num,
    test_size=0.2,
    random_state=2022
)

In [30]:
X_train

array([array([ 0.7850699 ,  0.7925065 ,  0.68391   ,  0.51949155,  3.9210072 ,
              -1.925423  ,  1.0364989 ,  3.7271247 , -1.1779671 , -2.379323  ,
               1.5550828 , -1.3587329 , -2.86917   ,  0.539684  ,  1.841728  ,
               4.9209185 ,  0.7290367 ,  1.2592117 ,  1.2734181 ,  1.7294807 ,
               0.9532299 , -1.363305  , -1.57219   , -1.0616881 , -0.21158302,
              -0.6155216 , -0.65631497, -0.8516976 ,  0.52584606,  0.40094995,
               0.43309093, -0.119858  ,  0.20791598, -2.604414  , -1.593533  ,
              -1.0798061 , -1.24694   ,  0.18778744, -0.5187036 , -0.521939  ,
              -0.01780796,  1.8656391 ,  0.497115  ,  1.082091  , -0.606477  ,
               0.06816723,  0.05442799, -0.9852489 , -0.70261294,  5.09947   ,
              -1.8852631 ,  0.83167   ,  1.562839  , -1.6896101 ,  1.2530651 ,
              -0.7321876 ,  1.375568  , -1.0148333 ,  1.228733  ,  0.15165202,
               1.080669  , -0.84440374,  1.2734786 ,

In [31]:
X_train.shape

(35918,)

In [27]:
X_test.shape

(8980,)

In [32]:
import numpy as np

X_train_2d=np.stack(X_train)
X_test_2d=np.stack(X_test)

X_train_2d


array([[ 0.7850699 ,  0.7925065 ,  0.68391   , ...,  0.311091  ,
        -1.3185085 ,  0.5586064 ],
       [-0.16637753,  3.120699  , -1.7741612 , ...,  0.6393125 ,
        -1.9734365 ,  1.5414468 ],
       [ 2.075564  , -2.4329736 ,  1.6463196 , ..., -1.3449867 ,
        -1.4542305 ,  0.3841053 ],
       ...,
       [ 0.0286357 ,  1.6997265 , -1.5029558 , ..., -0.6310583 ,
        -2.3059316 ,  1.8102632 ],
       [ 2.4272113 , -1.2375283 ,  1.5204664 , ..., -1.2258621 ,
        -0.4920283 ,  0.0332889 ],
       [-1.1633999 ,  0.9829463 , -3.6798668 , ..., -0.12614825,
        -2.2193022 ,  1.9722816 ]], dtype=float32)

In [34]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()
Scaled_train_embed=scaler.fit_transform(X_train_2d)
Scaled_test_embed=scaler.transform(X_test_2d)

clf=MultinomialNB()
clf.fit(Scaled_train_embed,y_train)

MultinomialNB()

In [35]:
from sklearn.metrics import classification_report

y_pred=clf.predict(Scaled_test_embed)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      4650
           1       0.97      0.97      0.97      4330

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [41]:
from sklearn.neighbors import KNeighborsClassifier

clf=KNeighborsClassifier(n_neighbors=5, metric='euclidean')
clf.fit(X_train_2d,y_train)
y_pred=clf.predict(X_test_2d)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      4650
           1       1.00      0.99      0.99      4330

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

